# Hate Speech Detection — Full Pipeline Demo

Runs the full 3-layer pipeline (SBERT retrieval → RAC classifier → LLM explanation) on custom input texts and exports a colour-coded HTML report.

In [ ]:
import sys, json, re, torch, faiss
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

sys.path.insert(0, str(Path("..").resolve()))  # adds src/ so retriever.py is importable
from retriever import retrieve_top_k, retrieve_top_k_above_threshold
from llm_explainer import explain, Layer2Output

## 2. LLM Backend

Choose your LLM backend. Groq is recommended (free tier, fast). Uncomment Option B or C for Ollama or OpenAI.

In [ ]:
# --- Option A: Groq (free, recommended) — set GROQ_API_KEY in your environment ---
# export GROQ_API_KEY="your_key_here"   (shell)  OR  set it in a .env file
import os
from groq import Groq

groq_key = os.environ.get("GROQ_API_KEY", "")

llm_client = Groq(api_key=groq_key)
LLM_MODEL  = "llama-3.3-70b-versatile"

# --- Option B: Ollama (local, free) ---
# import openai
# llm_client = openai.OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# LLM_MODEL = "mistral"

# --- Option C: OpenAI (paid) ---
# import openai
# llm_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY", ""))
# LLM_MODEL = "gpt-4o-mini"

print(f"LLM backend ready: {LLM_MODEL}")

LLM backend ready: llama-3.3-70b-versatile


## 3. Configuration

Choose the classifier model, index split and training dataset. `INDEX_SPLIT` and `DATASET` must match the saved weights.

In [ ]:
# Classifier and index configuration
MODEL_FAMILY = "roberta"   # "bert" | "roberta"
INDEX_SPLIT  = "full"  # "example" | "knowledge" | "full"
DATASET      = "ISHate"    # "IHC" | "ISHate"
K            = 3           # max neighbors to retrieve
THRESHOLD    = 0.5         # min cosine similarity (sbert is well-separated, not collapsed to ~0.997)

# Define paths
ROOT_DIR        = Path("../..")
INDEX_DIR       = ROOT_DIR / "corpus" / "index"
WEIGHTS_RAC_DIR = ROOT_DIR / "weigths" / "weights_rac_best_hyperparameters"

## 4. Custom Input Texts

Add the texts you want to analyse. No ground-truth labels needed.

In [ ]:
# Add or edit the texts to analyse
TEXTS = [
    {"id": 1, "text": "Write your first text here."},
    {"id": 2, "text": "Write your second text here."},
    # Add more entries as needed.
]

print(f"Loaded {len(TEXTS)} custom input(s).")
for t in TEXTS:
    print(f"  [{t['id']}] {t['text'][:100]}")


## 5. Load Pipeline

Load the SBERT retriever, FAISS index, and RAC classifier.

In [ ]:
# Retriever and classifier HF IDs
RETRIEVER_HF_ID = "sentence-transformers/all-mpnet-base-v2"

CLF_HF_IDS = {
    "bert":    "bert-base-uncased",
    "roberta": "roberta-base",
}

def load_pipeline(model_family, index_split, dataset):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # Retriever: always sbert, shared across all classifier configs
    print(f"Loading retriever: {RETRIEVER_HF_ID} ...")
    ret_tokenizer = AutoTokenizer.from_pretrained(RETRIEVER_HF_ID)
    ret_model     = AutoModel.from_pretrained(RETRIEVER_HF_ID).eval().to(device)

    index_path  = INDEX_DIR / f"vdb_{index_split}.faiss"
    lookup_path = INDEX_DIR / f"lookup_{index_split}.json"
    print(f"Loading index: {index_path} ...")
    index = faiss.read_index(str(index_path))
    with open(lookup_path) as f:
        documents = json.load(f)
    print(f"  Index size: {index.ntotal:,} vectors")

    # Classifier: model-family specific, trained on sbert-retrieved data
    clf_hf_id = CLF_HF_IDS[model_family]
    clf_path = str(WEIGHTS_RAC_DIR / model_family / "sbert" / index_split / dataset)
    print(f"Loading RAG classifier: {clf_path} ...")
    clf_tokenizer = AutoTokenizer.from_pretrained(clf_path)
    clf_model     = AutoModelForSequenceClassification.from_pretrained(clf_path).eval().to(device)

    print(f"\nReady: {model_family.upper()} | index=sbert/{index_split} | trained_on={dataset}")
    return ret_model, ret_tokenizer, index, documents, clf_model, clf_tokenizer, device


ret_model, ret_tokenizer, index, documents, clf_model, clf_tokenizer, device = load_pipeline(
    MODEL_FAMILY, INDEX_SPLIT, DATASET
)

## 6. Pipeline Helpers

`run_pipeline` runs Layer 2 (retrieval + classification) and Layer 3 (LLM explanation) for one text.

In [ ]:
# strip_label: remove [hate]/[not hate] prefix from a chunk text
_LABEL_RE = re.compile(r"^\[(hate|not hate)\]\s*:?\s*", re.IGNORECASE)

def strip_label(text):
    return _LABEL_RE.sub("", text).strip()


def run_pipeline(text, ret_model, ret_tokenizer, index, documents,
                 clf_model, clf_tokenizer, device,
                 llm_client, llm_model, k=K, threshold=THRESHOLD):

    # Layer 2a: retrieve neighbors
    retrieved = retrieve_top_k_above_threshold(
        text, threshold, ret_model, ret_tokenizer, index, documents, chunk_id=None, k=k
    )
    if not retrieved:  # fallback: nothing cleared the threshold
        retrieved = retrieve_top_k(
            text, ret_model, ret_tokenizer, index, documents, chunk_id=None, k=k
        )

    # Layer 2b: augment and classify
    sep = clf_tokenizer.sep_token or "[SEP]"
    augmented = f" {sep} ".join([text] + [t for t, _ in retrieved])
    inputs = clf_tokenizer(
        augmented, return_tensors="pt", truncation=True, padding=True, max_length=256
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = clf_model(**inputs).logits[0]
    probs      = F.softmax(logits, dim=-1)
    pred_idx   = torch.argmax(probs).item()
    label      = "hate" if pred_idx == 1 else "not hate"
    confidence = probs[pred_idx].item()

    # Layer 3: LLM explanation
    l2 = Layer2Output(
        original_text=text, label=label, confidence=confidence,
        hate_category="unknown", retrieved=retrieved
    )
    explanation = explain(l2, llm_client, llm_model)
    return l2, explanation


def display_result(example_id, text, l2, explanation):
    w = 80
    print("=" * w)
    print(f"[{example_id}]  TEXT : {text}")
    print("-" * w)
    print(f"LAYER 2  : {l2.label.upper()}  ({l2.confidence:.1%} confidence)")
    print()
    print(f"RETRIEVED NEIGHBORS ({len(l2.retrieved)}):")
    for i, (txt, score) in enumerate(l2.retrieved, 1):
        print(f"  [{i}] {score:.4f}  {strip_label(txt)[:100]}")
    print()
    print("LAYER 3 EXPLANATION:")
    print(f"  Summary   : {explanation.summary}")
    print(f"  Severity  : {explanation.severity}")
    print(f"  Action    : {explanation.recommended_action}")
    print(f"  Targets   : {', '.join(explanation.target_groups) if explanation.target_groups else chr(8212)}")
    print(f"  Evidence  : {explanation.evidence_used}")
    if explanation.moderator_note:
        print(f"  Note      : {explanation.moderator_note}")
    valid_str = "✓ passed" if explanation.validation_passed else "✗ FAILED (forced human-review)"
    print(f"  Validation: {valid_str}")
    print("=" * w)
    print()

## 7. Run Pipeline

Run the full pipeline on all input texts. Results are stored in `records`.

In [ ]:
import psutil, time, gc, numpy as np
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout
from llm_explainer import ExplainerOutput, _extract_label
from retriever import encode

print(f"RAM available: {psutil.virtual_memory().available / 1e9:.1f} GB")
print(f"Config : {MODEL_FAMILY.upper()} | index=sbert/{INDEX_SPLIT} | trained_on={DATASET}")
print(f"Running pipeline on {len(TEXTS)} input(s).\n")

LLM_TIMEOUT      = 30  # seconds per Groq call before giving up

# Unwrap IndexIDMap to extract raw vectors for numpy-based cosine search
# Unwrap IndexIDMap → extract raw vectors + ID map for pure-numpy cosine search
print("Extracting index vectors for numpy search...", end=" ", flush=True)
_inner  = faiss.downcast_index(index.index)
_xb     = np.empty((index.ntotal, index.d), dtype="float32")
_inner.reconstruct_n(0, index.ntotal, _xb)
_id_map = faiss.vector_to_array(index.id_map).astype("int64")
print(f"done  shape={_xb.shape}")

def retrieve_numpy(text, threshold, k, model, tokenizer):
    """Encode once with mean pooling (sbert), cosine sim via numpy — no FAISS at query time."""
    vec   = encode([text], model, tokenizer, batch_size=1, use_mean_pool=True)
    vec_n = vec / np.maximum(np.linalg.norm(vec, axis=1, keepdims=True), 1e-9)

    sims    = (_xb @ vec_n.T).squeeze()
    top_pos = np.argsort(sims)[::-1][:k]
    top_ids = _id_map[top_pos]
    scores  = sims[top_pos]

    retrieved = [
        (documents[str(int(cid))], float(sc))
        for cid, sc in zip(top_ids, scores)
        if sc >= threshold
    ][:k]
    if not retrieved:
        retrieved = [
            (documents[str(int(cid))], float(sc))
            for cid, sc in zip(top_ids, scores)
        ][:k]

    del vec, vec_n, sims, top_pos, top_ids, scores
    return retrieved

# Run pipeline on each input
records = []
t_start = time.time()

for i, entry in enumerate(TEXTS):
    text       = str(entry["text"])
    example_id = entry["id"]

    t0 = time.time()
    print(f"[{i+1:02d}/{len(TEXTS)}] id={example_id}  ...", end=" ", flush=True)

    retrieved = retrieve_numpy(text, THRESHOLD, K, ret_model, ret_tokenizer)

    sep       = clf_tokenizer.sep_token or "[SEP]"
    augmented = f" {sep} ".join([text] + [t for t, _ in retrieved])
    inputs    = clf_tokenizer(augmented, return_tensors="pt", truncation=True, padding=True, max_length=256)
    inputs    = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = clf_model(**inputs).logits[0]
    del inputs
    probs      = torch.nn.functional.softmax(logits, dim=-1)
    pred_idx   = torch.argmax(probs).item()
    label      = "hate" if pred_idx == 1 else "not hate"
    confidence = probs[pred_idx].item()
    del logits, probs

    l2 = Layer2Output(
        original_text=text, label=label, confidence=confidence,
        hate_category="unknown", retrieved=retrieved,
    )

    try:
        with ThreadPoolExecutor(max_workers=1) as ex:
            fut         = ex.submit(explain, l2, llm_client, LLM_MODEL)
            explanation = fut.result(timeout=LLM_TIMEOUT)
    except FuturesTimeout:
        print(f"TIMEOUT({LLM_TIMEOUT}s) ", end="", flush=True)
        explanation = ExplainerOutput(
            summary="LLM call timed out.", evidence_used=[], target_groups=[],
            severity="unknown", recommended_action="human-review",
            moderator_note="Groq call exceeded timeout.", validation_passed=False,
        )
    except Exception as e:
        print(f"ERR({e}) ", end="", flush=True)
        explanation = ExplainerOutput(
            summary=f"LLM error: {e}", evidence_used=[], target_groups=[],
            severity="unknown", recommended_action="human-review",
            moderator_note="LLM call raised an exception.", validation_passed=False,
        )

    elapsed = time.time() - t0
    print(f"pred={label.upper():8s}  conf={confidence:.1%}  {elapsed:.1f}s")

    records.append({
        "id"                : example_id,
        "text"              : text,
        "predicted"         : label,
        "confidence"        : round(confidence, 4),
        "n_retrieved"       : len(retrieved),
        "top_sim"           : round(retrieved[0][1], 4) if retrieved else None,
        "retrieved_passages": [
            {"text": strip_label(t), "label": _extract_label(t), "score": round(s, 4)}
            for t, s in retrieved
        ],
        "summary"           : explanation.summary,
        "evidence_used"     : explanation.evidence_used,
        "severity"          : explanation.severity,
        "action"            : explanation.recommended_action,
        "target_groups"     : explanation.target_groups,
        "moderator_note"    : explanation.moderator_note,
        "validation_passed" : explanation.validation_passed,
    })

    gc.collect()

results_df = pd.DataFrame(records)
total = time.time() - t_start
print(f"\nDone. {len(records)} input(s) processed  |  total={total/60:.1f} min")

## 8. Results Table

Display key fields for each input.

In [ ]:
# Display key fields for each input
print(f"Config : {MODEL_FAMILY.upper()} | index={INDEX_SPLIT} | trained_on={DATASET}")
print(f"Inputs : {len(records)}")
print("=" * 50)
display(results_df[["id", "predicted", "confidence", "n_retrieved", "top_sim", "severity", "action", "validation_passed"]])

## 9. Results Summary

Prediction and action breakdown across all inputs.

In [ ]:
print(f"\nResults for {len(records)} input(s):")
print(f"{'ID':<6} {'Predicted':<12} {'Confidence':<12} {'Action'}")
print("-" * 50)
for r in records:
    print(f"  {str(r['id']):<4}  {r['predicted'].upper():<12}  {r['confidence']:.1%:<12}  {r['action']}")

action_counts = {}
for r in records:
    action_counts[r["action"]] = action_counts.get(r["action"], 0) + 1
print(f"\nAction breakdown: {action_counts}")
hate_count = sum(1 for r in records if r["predicted"] == "hate")
print(f"Predicted hate: {hate_count}/{len(records)}")

## 10. Inspect Flagged Inputs

Filter results to explore specific predictions.

In [ ]:
# Adjust the filter to explore any subset of results

flagged = [r for r in records if r["predicted"] == "hate"]
print(f"Flagged as hate: {len(flagged)}/{len(records)}\n")
for r in flagged:
    print(f"[{r['id']}] pred={r['predicted'].upper():8s}  conf={r['confidence']:.1%}  action={r['action']}")
    print(f"{r['text'][:120]}")